In [2]:
import pandas as pd
import pymysql

In [5]:
conexion = pymysql.connect(
    host="dl-radar.cluster-ro-c7pmwdslewrp.us-east-1.rds.amazonaws.com",
    user = "debian",
    password= "eeAZU3v1FXCY9zmbvcS6kpEpyj",
    database="data_fact",
    port= 4408
)

cursor = conexion.cursor()

In [6]:
query = """
SELECT *
FROM base_rucs_sri;
"""
base_registro_civil = pd.read_sql_query(query, conexion)

C:\Users\anali\AppData\Local\Temp\ipykernel_29500\631102470.py:5: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  base_registro_civil = pd.read_sql_query(query, conexion)


In [3]:
base_registro_civil = pd.read_parquet(r"C:\Users\anali\OneDrive - PUBLIPROMUEVE S.A\Ruben Freire's files - CENTROS COMERCIALES\sandbox\bases\base_rucs_sri.parquet")

In [34]:
base_registro_civil[(base_registro_civil['nombre_fantasia_comercial'].str.contains("quic", case = False, na = False)) & (base_registro_civil['razon_social'].str.contains("Management", case = False, na = False))]

,id_establecimiento,numero_ruc,numero_establecimiento,razon_social,nombre_fantasia_comercial,cod_estado_contribuyente,estado_contribuyente,cod_estado_establecimiento,estado_establecimiento,matriz,...,direccion_completa,fecha_inicio_actividades_comercio,fecha_cese_comercio,fecha_reinicio_actividades_comercio,fecha_actualizacion_comercio,nombre_representante_legal,identificacion_representante_legal,representantes_legales,fecha_actualizacion,encontrado
7173678,1.791278e+15,1.791278e+12,5.0,DK MANAGEMENT SERVICES S.A.,quicentro sur,1.0,ACTIVO,1.0,ABIERTO,0,...,PICHINCHA / QUITO / CHILLOGALLO / AV. MORAN VA...,1994-09-05,None,None,2026-01-13,CIMA-MANAGEMENT S.A.S.,1793200384001,"[{""nombre"": ""CIMA-MANAGEMENT S.A.S."", ""identif...",2026-01-30 07:53:02,1
7173680,1.791278e+15,1.791278e+12,7.0,DK MANAGEMENT SERVICES S.A.,quicentro shopping,1.0,ACTIVO,1.0,ABIERTO,0,...,PICHINCHA / QUITO / IÑAQUITO / AVENIDA DE LOS ...,1994-09-05,None,None,2026-01-13,CIMA-MANAGEMENT S.A.S.,1793200384001,"[{""nombre"": ""CIMA-MANAGEMENT S.A.S."", ""identif...",2026-01-30 07:53:02,1


In [4]:
base_registro_civil['nombre_fantasia_comercial'] = (
    base_registro_civil['nombre_fantasia_comercial']
        .replace("", pd.NA)
)


In [5]:
base_registro_civil = base_registro_civil[base_registro_civil["nombre_fantasia_comercial"].notna()] 

In [6]:
base_registro_civil['motivo_cancelacion_suspension'] = (base_registro_civil['motivo_cancelacion_suspension'].replace("", pd.NA))

In [7]:
base_registro_civil = base_registro_civil[base_registro_civil['motivo_cancelacion_suspension'].isna()]

In [8]:
recreo_nombres = pd.read_excel(r"C:\Users\anali\OneDrive - PUBLIPROMUEVE S.A\Ruben Freire's files - CENTROS COMERCIALES\ccrecreo.xlsx")

In [9]:
dicc_nombre_fantasia = set(recreo_nombres['LOCAL'])

In [10]:
import unicodedata
import re

def normalizar(s):
    if pd.isna(s):
        return ""
    s = s.lower()
    s = unicodedata.normalize('NFD', s)
    s = ''.join(c for c in s if unicodedata.category(c) != 'Mn')
    s = re.sub(r'[^a-z0-9 ]', ' ', s)
    s = re.sub(r'\s+', ' ', s).strip()
    return s


In [11]:
def qgrams(s, q=3):
    return {s[i:i+q] for i in range(len(s) - q + 1)}


In [12]:
def jaccard(a, b):
    if not a or not b:
        return 0.0
    return len(a & b) / len(a | b)


In [13]:
dic_norm = {normalizar(x): x for x in dicc_nombre_fantasia}

dic_qgrams = {
    k: qgrams(k, q=3)
    for k in dic_norm.keys()
}


In [73]:
def match_qgram(nombre, threshold=0.6):
    s = normalizar(nombre)
    q_s = qgrams(s)

    mejor, score = None, 0
    for k, q_k in dic_qgrams.items():
        sim = jaccard(q_s, q_k)
        if sim > score:
            mejor, score = dic_norm[k], sim

    if score >= threshold:
        return mejor, score
    return None, score

In [74]:
base_registro_civil['nombre_fantasia_comercial'] = base_registro_civil['nombre_fantasia_comercial'].apply(normalizar) 

In [75]:
base_registro_civil[['provincia',  'canton', 'parroquia', 'calles']] = (
    base_registro_civil['direccion_completa']
        .str.split('/', n=3, expand=True)
)


In [76]:
mask_canton = base_registro_civil['canton'].str.contains("Quito", case = False, na = False) 
mask_parroquia = base_registro_civil['parroquia'].str.contains("Magdalena", case = False, na = False)
mask_calle_mald  = base_registro_civil['calles'].str.contains("MALDONADO", case = False, na = False)
mask_calles = base_registro_civil['calles'].str.contains("S11C|OE2F", case = False, na = False)

mask_general = (mask_canton & mask_parroquia & mask_calle_mald) | mask_calles

In [77]:
base_registro_civil_test = base_registro_civil[mask_general]

In [78]:
base_registro_civil_test[['match_qgram', 'score_qgram']] = (
    base_registro_civil_test['nombre_fantasia_comercial']
      .apply(lambda x: pd.Series(match_qgram(x)))
)

C:\Users\anali\AppData\Local\Temp\ipykernel_20172\404753551.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  base_registro_civil_test[['match_qgram', 'score_qgram']] = (
C:\Users\anali\AppData\Local\Temp\ipykernel_20172\404753551.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  base_registro_civil_test[['match_qgram', 'score_qgram']] = (


In [79]:
base_registro_civil_test = base_registro_civil_test.sort_values(by = 'score_qgram', ascending = False)

In [81]:
base_registro_civil_test[(base_registro_civil_test['match_qgram'].notna())][['razon_social','direccion_completa', 'nombre_fantasia_comercial', 'match_qgram', 'score_qgram']]

,razon_social,direccion_completa,nombre_fantasia_comercial,match_qgram,score_qgram
5933912,CRUZ ORTEGA JULY,PICHINCHA / QUITO / LA MAGDALENA / AV MALDONAD...,lluvia d bendiciones,LLUVIA D´ BENDICIONES,1.000000
5934116,PEREZ MORALES RUTH ELIZABETH,PICHINCHA / QUITO / LA MAGDALENA / AV. PEDRO V...,shiatsu masajes,SHIATSU MASAJES,1.000000
5965687,MARTINEZ BASTIDAS ROBINSON JAIRO,PICHINCHA / QUITO / LA MAGDALENA / AV MALDONAD...,palacio del cinturon,PALACIO DEL CINTURÓN,1.000000
5975117,MARTINEZ MONTENEGRO DIEGO PAUL,PICHINCHA / QUITO / LA MAGDALENA / AV. MALDONA...,abismo,ABISMO,1.000000
5916145,HEREDIA TAYO JUAN FRANCISCO,PICHINCHA / QUITO / LA MAGDALENA / AV. MALDONA...,mobile solutions,MOBILE SOLUTIONS,1.000000
...,...,...,...,...,...
2426657,ORTEGA CHOEZ MARIA CAROLINA,PICHINCHA / QUITO / LA MAGDALENA / AV MALDONAD...,impulsamotos,IMPULSA MOTOS,0.615385
2232688,VALLADARES BACO FERNANDO ARMANDO,PICHINCHA / QUITO / LA MAGDALENA / AV. PEDRO V...,bubble tea,BUBBLE TEA ISLA,0.615385
7387823,TARTAS VASCAS S.A.S.,PICHINCHA / QUITO / LA MAGDALENA / AV. MALDONA...,tartas de la chef caro,TARTA VASCA DE LA CHEF CARO,0.607143
7182202,CORPMUNAB SOCIEDAD ANONIMA,PICHINCHA / QUITO / LA MAGDALENA / AV MALDONAD...,mabel,MABELEN,0.600000
